In [1]:

import re


nodes = 10
counts = [0 for _ in range(nodes)]
counts_to_last = [0 for _ in range(nodes)]
counts_to_share = [0 for _ in range(nodes)]

# this is the pattern to look for when a model is in aggregation and actively sending model data over to a peer.
# when Weighting 0 it means that it is instead sharing the model to someone as is not actually in aggregation
model_transfer_overhead_pattern = r"\[recieve_model\] Recieving Model Data: Length 44807785 and Weighting ([1-9]|10)$"
# this pattern is to ensure that there are definitely 20 of these messages to ensure that the model does converge to single point for 20 communication rounds
model_transfer_overhead_pattern_10 = r"\[recieve_model\] Recieving Model Data: Length 44807785 and Weighting 10$"
# this pattern is for every time the model is sent with weight 0 which indicates it is because of sharing
model_transfer_overhead_pattern_sharing = r"\[send_model\] Sending Model Data: Length 44807785 and Weighting 0$"

for i in range(nodes):
    file = open(f"./podman-logs/10-nodes/node_{i+1}_logs.txt")
    for line in file:
        if re.search(model_transfer_overhead_pattern,line.strip()):
            counts[i] += 1
        if re.search(model_transfer_overhead_pattern_10,line.strip()):
            counts_to_last[i] += 1
        if re.search(model_transfer_overhead_pattern_sharing,line.strip()):
            counts_to_share[i] += 1
    file.close()


print(f"Models recieved by each node: {counts}")
tally_last = 0
for count in counts_to_last:
    tally_last += count
print(f"Each time node has been last: {counts_to_last}")
print(f"Time it has Aggregated to finish: {tally_last}")
tally_share = 0
for count in counts_to_share:
    tally_share += count
print(f"Amount of times the node has shared {counts_to_share}")
print(f"Total shares {tally_share}")

Models recieved by each node: [34, 34, 40, 34, 30, 26, 42, 41, 32, 47]
Each time node has been last: [3, 3, 1, 1, 1, 0, 2, 4, 0, 5]
Time it has Aggregated to finish: 20
Amount of times the node has shared [39, 28, 10, 17, 9, 1, 20, 39, 4, 56]
Total shares 223


In [2]:
from numpy import average

biggest_agg = max(counts)
averages_agg = average(counts)
minmal_agg = min(counts)
biggest_sh = max(counts_to_share)
averages_sh = average(counts_to_share)
minmal_sh = min(counts_to_share)
print(f"maximum model transfers of a peer aggregating : {biggest_agg}")
print(f"average model transfers of a peer aggregating : {averages_agg}")
print(f"minimum model transfers of a peer aggregating : {minmal_agg}")
print(f"maximum model transfers of a peer sharing : {biggest_sh}")
print(f"average model transfers of a peer sharing : {averages_sh}")
print(f"minimum model transfers of a peer sharing : {minmal_sh}")

maximum model transfers of a peer aggregating : 47
average model transfers of a peer aggregating : 36.0
minimum model transfers of a peer aggregating : 26
maximum model transfers of a peer sharing : 56
average model transfers of a peer sharing : 22.3
minimum model transfers of a peer sharing : 1


In [3]:
# the amount of communication rounds is 20 for 100 epochs where each communcation round is 5 epochs
rounds = 20

print(f"maximum model transfers per round (agg) : {biggest_agg/rounds}")
print(f"average model transfers per round (agg) : {averages_agg/rounds}")
print(f"minimum model transfers per round (agg) : {minmal_agg/rounds}")
print(f"maximum model transfers per round (sh) : {biggest_sh/rounds}")
print(f"average model transfers per round (sh) : {averages_sh/rounds}")
print(f"minimum model transfers per round (sh) : {minmal_sh/rounds}")

maximum model transfers per round (agg) : 2.35
average model transfers per round (agg) : 1.8
minimum model transfers per round (agg) : 1.3
maximum model transfers per round (sh) : 2.8
average model transfers per round (sh) : 1.115
minimum model transfers per round (sh) : 0.05


In [4]:
from math import log

# the 2*log2(10) target
# why 2*log2(10)? because it should account for both aggregation being log2(10) AND sharing being log2(10)
print(f"Olog(N) target max communication overhead : {log(nodes,2)}")

Olog(N) target max communication overhead : 3.3219280948873626


In [5]:
print(f"Under O(LogN) (agg) : {biggest_agg/rounds<log(nodes,2)}")
print(f"Under O(LogN) (sh) : {biggest_sh/rounds<log(nodes,2)}")
print(f"Under O(LogN) (agg) :\n actual max: {biggest_agg/rounds} theoritical max: {log(nodes,2)}")
print(f"Under O(LogN) (sh) :\n actual max: {biggest_sh/rounds} theoritical max: {log(nodes,2)}")

Under O(LogN) (agg) : True
Under O(LogN) (sh) : True
Under O(LogN) (agg) :
 actual max: 2.35 theoritical max: 3.3219280948873626
Under O(LogN) (sh) :
 actual max: 2.8 theoritical max: 3.3219280948873626
